# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [6]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import (
    load_and_clean_data, 
    calculate_profitable_trades, 
    analyze_entry_timing, 
    display_profitable_strategies,
    Strategy,
    create_strategy_library,
    evaluate_all_strategies,
    get_top_strategies,
    get_top_strategies_by_edge,
    analyze_pullback_profitability
)

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

# Display profitable trades analysis
display(HTML("""
    <h2>Profitable Trades</h2>
    <p>What are simple trading ideas that are profitable?</p>
"""))
display(calculate_profitable_trades(df))
display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>CH trades are out of question, they're not profitable.</li>
        <li>3 pips extra is harming the strategy, while 1 pip can improve it slightly.</li>
        <li>Overall any filtration is harmful.</li>
    </ul>
"""))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra 1 pip,43.3%,26.0%,18.3%
1,With Extra 2 pips,43.3%,26.9%,14.4%
2,Total,38.5%,25.0%,20.2%
3,With Extra 3 pips,37.5%,22.1%,13.5%
4,Just BOS Trades,25.0%,17.3%,14.4%
5,With EMA Direction,23.1%,14.4%,11.5%
6,With EMA + BOS,17.3%,11.5%,9.6%
7,Against EMA Direction,15.4%,10.6%,8.7%
8,Just CH Trades,13.5%,7.7%,5.8%
9,With EMA + CH,5.8%,2.9%,1.9%


In [7]:
# Display entry timing analysis using the imported function
display(HTML("""
    <h2>When To Enter</h2>
    <p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>
"""))
display(analyze_entry_timing(df))
display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>Very interesting that here, my strategy with 1M confirmation candle is the worse.</li>
"""))
display(HTML("""
    <p><b>Open Questions</b></p>
    <ul>
        <li>That 5M Stop entry makes a lot of sense as all profitable strategies would have it, but in reality somehow it performs the worse. Why?</li>
        <li>Why does 1M confirmation candle entry outperforms all other entries in the next setups analysis?</li>
    </ul>
"""))

,Idea,Notation,Win Rate,With Extra,With Extra & 1:3 RRR
0,5M Stop,33W - 35L,48.5%,55.9%,27.9%
1,5M Breakout,32W - 37L,46.4%,53.6%,29.0%
2,5M Confirmation Candle,33W - 42L,44.0%,50.7%,28.0%
3,1M Confirmation Candle,40W - 64L,38.5%,43.3%,22.1%


In [8]:
# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

# Add all strategies from the library
strategies.extend(create_strategy_library())

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

# Display top performing strategies for each RRR - sorted by Edge
display(HTML("<h2>🎯 TOP Performing Strategies</h2>"))

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies by Edge</h3>"))
    
    # Get and display top strategies sorted by edge
    top_df = get_top_strategies_by_edge(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px'}
    ).set_properties(
        subset=['Edge'],
        **{'color': 'green'}
    )
    
    display(styled_df)
    print()  # Add spacing

display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>EMA + BOS is on 5th place. So anything above it should be tradeable</li>
        <li>According this list, trading all BOS trades would simplify things a lot. Lets try that out!</li>
    </ul>
"""))

,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + SL > 5 pips,1M CC,18,12,6,66.7%,16.7%,6R
1,30M Trend + 5 < SL < 15,1M CC,18,12,6,66.7%,16.7%,6R
2,30M Trend + SL > 5 pips,5M CC,18,12,6,66.7%,16.7%,6R
3,30M Trend + 5 < SL < 15,5M CC,18,12,6,66.7%,16.7%,6R
4,30M Trend + SL > 5 pips,5M Breakout,18,12,6,66.7%,16.7%,6R
5,30M Trend + 5 < SL < 15,5M Breakout,18,12,6,66.7%,16.7%,6R
6,30M Trend + SL > 5 pips,5M Stop,18,11,7,61.1%,11.1%,4R
7,30M Trend + 5 < SL < 15,5M Stop,18,11,7,61.1%,11.1%,4R
8,Aggressive: SL >= 7 pips,1M CC,17,10,7,58.8%,8.8%,3R
9,30M Trend + News > 2hrs,1M CC,12,7,5,58.3%,8.3%,2R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,8,4,4,50.0%,16.7%,4R
1,30M Trend + News > 2hrs,1M CC,12,6,6,50.0%,16.7%,6R
2,30M Trend + News > 2hrs,5M CC,12,6,6,50.0%,16.7%,6R
3,30M Trend + News > 2hrs,5M Stop,12,6,6,50.0%,16.7%,6R
4,30M Trend + SL > 5 pips,5M Breakout,18,9,9,50.0%,16.7%,9R
5,30M Trend + 5 < SL < 15,5M Breakout,18,9,9,50.0%,16.7%,9R
6,30M Trend + News > 2hrs,5M Breakout,12,6,6,50.0%,16.7%,6R
7,News in 2+ Hours,1M CC,26,12,14,46.2%,12.9%,10R
8,30M Trend + BOS,1M CC,42,19,23,45.2%,11.9%,15R
9,30M Trend + BOS + SL < 10,1M CC,42,19,23,45.2%,11.9%,15R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,8,4,4,50.0%,25.0%,8R
1,30M Trend + News > 2hrs,1M CC,12,5,7,41.7%,16.7%,8R
2,30M Trend + News > 2hrs,5M CC,12,5,7,41.7%,16.7%,8R
3,30M Trend + News > 2hrs,5M Breakout,12,5,7,41.7%,16.7%,8R
4,30M Trend + SL > 5 pips,5M CC,18,7,11,38.9%,13.9%,10R
5,30M Trend + 5 < SL < 15,5M CC,18,7,11,38.9%,13.9%,10R
6,News in 2+ Hours,5M CC,26,10,16,38.5%,13.5%,14R
7,30M Trend + BOS,1M CC,42,16,26,38.1%,13.1%,22R
8,30M Trend + BOS + SL < 10,1M CC,42,16,26,38.1%,13.1%,22R
9,BOS + Conservative SL,5M Stop,8,3,5,37.5%,12.5%,4R


In [9]:
# Pullback Analysis
display(HTML("<h2>Pullback Impact on Profitability</h2>"))
display(HTML("<p>How does pullback size affect trade profitability? Analyzing trades where TP > SL.</p>"))

# Display pullback profitability analysis
pullback_analysis = analyze_pullback_profitability(df)
display(pullback_analysis)

# Add summary
display(HTML("""
<p><b>Summary</b></p>
<ul>
    <li>According to this analysis, pullback of 0.5 pip should not harm the strategy but could dramatically reduce the the path to the TP.</li>
</ul>
"""))

,Pullback,All Trades,Profitable Trades,Percentage
0,All (No Filter),104,53,51.0%
1,0.5 pips,97,48,49.5%
2,1.0 pip,90,41,45.6%
3,1.5 pips,84,36,42.9%
4,2.0 pips,71,30,42.3%
5,2.5 pips,63,23,36.5%
6,3.0 pips,57,20,35.1%


In [10]:
# Display only profitable strategies
display_profitable_strategies(strategy_results)

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,104,104,104
1,Wins,53,36,29
2,Losses,51,68,75
3,Win Rate,51.0%,34.6%,27.9%
4,Edge,1.0%,1.3%,2.9%
5,Outcome,2R,4R,12R
6,Entry,1M CC,1M CC,1M CC


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,41,41,41
1,Wins,22,15,13
2,Losses,19,26,28
3,Win Rate,53.7%,36.6%,31.7%
4,Edge,3.7%,3.3%,6.7%
5,Outcome,3R,4R,11R
6,Entry,1M CC,1M CC,1M CC


,BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,66,66,66
1,Wins,35,26,22
2,Losses,31,40,44
3,Win Rate,53.0%,39.4%,33.3%
4,Edge,3.0%,6.1%,8.3%
5,Outcome,4R,12R,22R
6,Entry,1M CC,1M CC,1M CC


,Moderate Risk: SL 3-6 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,58,58,58
1,Wins,30,19,14
2,Losses,28,39,44
3,Win Rate,51.7%,32.8%,24.1%
4,Edge,1.7%,-0.5%,-0.9%
5,Outcome,2R,-1R,-2R
6,Entry,1M CC,1M CC,1M CC


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,10,5,4
2,Losses,7,12,13
3,Win Rate,58.8%,29.4%,23.5%
4,Edge,8.8%,-3.9%,-1.5%
5,Outcome,3R,-2R,-1R
6,Entry,1M CC,1M CC,1M CC


...

,30M Trend + News > 2hrs,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,12,12,12
1,Wins,7,6,4
2,Losses,5,6,8
3,Win Rate,58.3%,50.0%,33.3%
4,Edge,8.3%,16.7%,8.3%
5,Outcome,2R,6R,4R
6,Entry,5M Stop,5M Stop,5M Stop


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,9,6,4
2,Losses,8,11,13
3,Win Rate,52.9%,35.3%,23.5%
4,Edge,2.9%,2.0%,-1.5%
5,Outcome,1R,1R,-1R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,18,18,18
1,Wins,12,9,6
2,Losses,6,9,12
3,Win Rate,66.7%,50.0%,33.3%
4,Edge,16.7%,16.7%,8.3%
5,Outcome,6R,9R,6R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,18,18,18
1,Wins,12,9,6
2,Losses,6,9,12
3,Win Rate,66.7%,50.0%,33.3%
4,Edge,16.7%,16.7%,8.3%
5,Outcome,6R,9R,6R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + News > 2hrs,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,12,12,12
1,Wins,7,6,5
2,Losses,5,6,7
3,Win Rate,58.3%,50.0%,41.7%
4,Edge,8.3%,16.7%,16.7%
5,Outcome,2R,6R,8R
6,Entry,5M Breakout,5M Breakout,5M Breakout
